### Prediction Steps

In [93]:
import pandas as pd
import pickle
from tensorflow.keras.models import load_model

### Load Models

In [94]:
with open("gender_label_encoder.pkl", "rb") as file:
    gender_label_encoder = pickle.load(file)
    
with open("geo_one_hot_encoder.pkl", "rb") as file:
    geo_one_hot_encoder = pickle.load(file)
    
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)
    
sequential_model_h5 = load_model("sequential_model.h5")
sequential_model_keras = load_model("sequential_model.keras")

c:\Users\khem6\Desktop\Training\gen-ai\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop_5', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(store)


In [95]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [96]:
input_data_df = pd.DataFrame([input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


### Input processing 

In [97]:
input_data_df["Gender"] = gender_label_encoder.transform(input_data_df["Gender"])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [98]:
geo_data = geo_one_hot_encoder.transform([ input_data_df["Geography"] ])

c:\Users\khem6\Desktop\Training\gen-ai\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [99]:
print(geo_one_hot_encoder.feature_names_in_)
print(geo_one_hot_encoder.get_feature_names_out())

['Geography']
['Geography_France' 'Geography_Germany' 'Geography_Spain']


In [100]:
geo_data.toarray()

array([[1., 0., 0.]])

In [101]:
geo_data_df = pd.DataFrame(columns=geo_one_hot_encoder.get_feature_names_out(), data=geo_data.toarray())
geo_data_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [102]:
input_data_df = pd.concat([ input_data_df.drop("Geography", axis=1), geo_data_df ], axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


### Scaling the input data

In [105]:
input_scaled = scaler.transform(input_data_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

### Predict churn

In [106]:
prediction=sequential_model_h5.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step


array([[0.15076435]], dtype=float32)

In [108]:
prob = prediction[0][0]
prob

np.float32(0.15076435)

In [109]:
if prob > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.


In [110]:
prediction=sequential_model_keras.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step


array([[0.15076435]], dtype=float32)

In [112]:
prediction[0][0]

np.float32(0.15076435)